<!-- nav -->
[← 8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb) · [Home](../../README.md) · [8.15 · SystemVerilog Assertions →](08_15_SystemVerilog_Assertions.ipynb)

# 8.14 · SystemVerilog Patterns

Tutorial 8 showed a `matches` comparing a value with a pattern, `v matches '{.a, .b}`, and binding the pattern's
variables (08.2). This page takes the patterns themselves, IEEE 1800's pattern matching (12.6), as their dialect,
`SystemVerilog Patterns`, models them: from the smallest, a variable and a wildcard, to tagged and structure patterns,
each in the builders' fluent calls beside the text SystemVerilog writes. Then what is done with them: what each binds;
validated, with one refused; held by a `matches`, which binds them in a conditional's consequence; and never
evaluated, each kind refused by name. Last, the editions that have them.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4); `variable` writes a variable
pattern, `.x`, of its spelling, as tutorial 5's `variable` writes a name; `binds` lists the names a pattern binds, and
`refused` gives what an evaluator says of an expression, its value or the reason it refuses it, as tutorial 8's
does.

In [1]:
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog, Patterns as SVPatterns
from mbse.Expressions.Framework import Domains, Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
Patterns = SVPatterns.Expressions.DIALECT  # the dialect, as the framework reads it
name = lambda spelling: lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling(spelling))))
literal = lambda value: lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(value)))
variable = lambda spelling: lambda p: p.variablepattern(lambda v: v.name(lambda i: i.spelling(spelling)))  # .spelling


def binds(expression):
    """The names a pattern binds, in order."""
    return ", ".join(sorted(SVPatterns.Expressions.bound_names(Terms.term(expression)))) or "none"


def refused(evaluate, expression):
    """What `evaluate` says of `expression`: its value, or the reason it refuses it."""
    try:
        return str(evaluate(expression))
    except (NotImplementedError, ValueError) as error:
        return str(error)

<a id="T08.14.1"></a>
## 08.14.1 · A variable and a wildcard

```text
.x        .*
```

The smallest patterns match anything. `.x` is a variable pattern, `variablepattern`: it matches any value and binds
the name `x` to it, its name `Expressions`' identifier helper, the one a name expression holds (tutorial 4). `.*` is a
wildcard, `wildcardpattern`, which matches any value and binds nothing. Each is a term of `SystemVerilog Patterns`, the
root of its holder, `S.SystemVerilog.Patterns.Expression`, and the family's printer writes it:

In [2]:
with store.transaction():
    smallest = [S.SystemVerilog.Patterns.Expression(lambda e: e.root(variable("x"))),
                S.SystemVerilog.Patterns.Expression(lambda e: e.root(lambda p: p.wildcardpattern(lambda w: w)))]
for pattern in smallest:
    form = Terms.form(Terms.term(pattern))
    print(f"{SystemVerilog.Text.ToText(pattern):3}", f"{form.kind:16}", "binds", f"{binds(pattern):5}", Terms.dialect_of(Terms.term(pattern)))
assert [SystemVerilog.Text.ToText(p) for p in smallest] == [".x", ".*"]

.x  variablepattern  binds x     Expressions.SystemVerilog.Patterns.Term
.*  wildcardpattern  binds none  Expressions.SystemVerilog.Patterns.Term


<a id="T08.14.2"></a>
## 08.14.2 · Tagged, structure and parenthesized patterns

```text
tagged Valid .x        tagged None
'{.a, .b}              '{a: .x, b: 1}
(tagged Valid .x)
```

A tagged union's value is matched by its member: `tagged Valid .x` matches the union when `Valid` is its member, and
matches the member's value with `.x`; `tagged None` names a member with no value. A structure is matched member by
member, in order, `'{.a, .b}`, or by name, `'{a: .x, b: 1}`, each named member a `patternmember`. Where a pattern is an
expression, `1` here, it is a constant pattern, which matches a value equal to it: no kind of its own, but
`Expressions`' own literal, since the dialect's `Term` lists `Expressions`' branches (tutorial 8, 08.3). Parentheses
written around a pattern are kept, a `parenthesizedpattern`. Patterns nest, and what a pattern binds is every variable
within it:

In [3]:
valid = lambda p: p.taggedpattern(lambda t: t.member(lambda i: i.spelling("Valid")).pattern(variable("x")))
with store.transaction():
    nested = [S.SystemVerilog.Patterns.Expression(lambda e, root=root: e.root(root)) for root in [
        valid,
        lambda p: p.taggedpattern(lambda t: t.member(lambda i: i.spelling("None"))),
        lambda p: p.structurepattern(lambda s: s.items([variable("a"), variable("b")])),
        lambda p: p.structurepattern(lambda s: s.items([
            lambda m: m.patternmember(lambda p: p.name(lambda i: i.spelling("a")).pattern(variable("x"))),
            lambda m: m.patternmember(lambda p: p.name(lambda i: i.spelling("b")).pattern(literal(1)))])),
        lambda p: p.parenthesizedpattern(lambda q: q.pattern(valid)),
    ]]
for pattern in nested:
    form = Terms.form(Terms.term(pattern))
    print(f"{SystemVerilog.Text.ToText(pattern):18}", f"{form.kind:20}", "binds", f"{binds(pattern):5}", "| its terms:",
          ", ".join(Terms.kind_of(a)[0] for a in form.arguments if a is not None) or "none")
assert [SystemVerilog.Text.ToText(p) for p in nested] == [
    "tagged Valid .x", "tagged None", "'{.a, .b}", "'{a: .x, b: 1}", "(tagged Valid .x)"]

tagged Valid .x    taggedpattern        binds x     | its terms: variablepattern
tagged None        taggedpattern        binds none  | its terms: none
'{.a, .b}          structurepattern     binds a, b  | its terms: variablepattern, variablepattern
'{a: .x, b: 1}     structurepattern     binds x     | its terms: patternmember, patternmember
(tagged Valid .x)  parenthesizedpattern binds x     | its terms: taggedpattern


A pattern binds, but it is no binder of its own: none of its kinds refines the framework's `Let` (tutorial 7). It binds
through what holds it, a `matches` (08.14.4).

<a id="T08.14.3"></a>
## 08.14.3 · Validated, and one refused

`validate` checks a pattern as it checks any term: what each kind requires, its identifiers' spellings, and the names its
constant patterns read, which are references, not bindings, so the caller must bind them. A structure pattern's members
are all in order or all by name (IEEE 1800 12.6), so one that mixes them is refused:

In [4]:
with store.transaction():
    constant, mixed, misspelled = [S.SystemVerilog.Patterns.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda p: p.structurepattern(lambda s: s.items([
            variable("a"), lambda e: e.binaryexpression(lambda o: o.left(name("n")).operator("+").right(literal(1)))])),
        lambda p: p.structurepattern(lambda s: s.items([
            lambda m: m.patternmember(lambda p: p.name(lambda i: i.spelling("a")).pattern(variable("x"))), variable("y")])),
        variable("1x"),
    ]]
checked = [(constant, ()), (constant, {"n"}), (mixed, ()), (misspelled, ())]
for pattern, bound in checked:
    print(f"{SystemVerilog.Text.ToText(pattern):13}", f"{', '.join(bound) or '-':3}", "; ".join(Patterns.validate(pattern, bound)) or "valid")
assert Patterns.validate(checked[2][0]) == ["a StructurePattern's members are in order or by name, not both"]

'{.a, n + 1}  -   argument 1: left: nameexpression 'n' is not bound
'{.a, n + 1}  n   valid
'{a: .x, .y}  -   a StructurePattern's members are in order or by name, not both
.1x           -   '1x' is not an identifier


<a id="T08.14.4"></a>
## 08.14.4 · Held by a `matches`

```text
v matches tagged Valid .x ? x + 1 : 0
v matches '{.a, .b} &&& a > b ? a : b
```

A pattern stands in one place in SystemVerilog's expressions: a `matches`' pattern, the one property of `Expressions`'
kinds that holds this dialect's `Term`, read from the store. A `matches` binds its pattern's variables where its
conditional is true: in the consequence, and in the conditions after it in a `&&&` (12.6.2), never in the value it
matches or in the alternative. So `x` is bound in `x + 1`, and only `v` is free; in the second, `a > b` reads what the
pattern bound:

In [5]:
union = store.registered(Patterns.union)
host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches
print(", ".join(f"{b.name}.{p}" for b in host for p, prop in b.schema.properties.items()
                if union in (prop.schema, getattr(prop.schema, "element", None))))
with store.transaction():
    chosen = [S.SystemVerilog.Expressions.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.conditionalexpression(lambda c: c
            .condition(lambda d: d.matchesexpression(lambda m: m.value(name("v")).pattern(valid)))
            .consequence(lambda d: d.binaryexpression(lambda b: b.left(name("x")).operator("+").right(literal(1))))
            .alternative(literal(0))),
        lambda u: u.conditionalexpression(lambda c: c
            .condition(lambda d: d.predicateexpression(lambda p: p.conditions([
                lambda d: d.matchesexpression(lambda m: m.value(name("v")).pattern(
                    lambda p: p.structurepattern(lambda s: s.items([variable("a"), variable("b")])))),
                lambda d: d.binaryexpression(lambda b: b.left(name("a")).operator(">").right(name("b")))])))
            .consequence(name("a")).alternative(name("b"))),
    ]]
dialect = SystemVerilog.Expressions.DIALECT
for expression in chosen:
    print(f"{SystemVerilog.Text.ToText(expression):38}", "free:", ", ".join(sorted(Symbolics.free(expression, dialect))),
          "| problems:", "; ".join(dialect.validate(expression, {"v"})) or "none")
assert [SystemVerilog.Text.ToText(e) for e in chosen] == [
    "v matches tagged Valid .x ? x + 1 : 0", "v matches '{.a, .b} &&& a > b ? a : b"]

matchesexpression.pattern
v matches tagged Valid .x ? x + 1 : 0  free: v | problems: none
v matches '{.a, .b} &&& a > b ? a : b  free: b, v | problems: alternative: nameexpression 'b' is not bound


The second's alternative reads `b`, which the pattern did not bind there: the conditional's scopes say so, and
`validate`, given `v` alone, reports the alternative's `b` as unbound, in the words it reports any free name with.

<a id="T08.14.5"></a>
## 08.14.5 · Never evaluated

SystemVerilog's evaluator matches no pattern (stage 6, "Q11"): it refuses each of the dialect's kinds by name, held alone
or by a `matches`, and a conditional sizes its condition first, so the `matches` is refused by its own name before its
consequence, which reads what the pattern binds, is read. A pattern has no domain to infer, `any`:

In [6]:
evaluate = SystemVerilog.Evaluators.OfAny
for pattern in [*smallest, *nested[:3]]:
    print(f"{SystemVerilog.Text.ToText(pattern):16}", refused(evaluate, pattern))
print(f"{SystemVerilog.Text.ToText(chosen[0]):16}", refused(lambda e: evaluate(e, {"v": 1}), chosen[0]))
print("inferred:", Patterns.infer(nested[0]).name())
assert Patterns.infer(nested[0]) is Domains.Anything

.x               SystemVerilog Patterns variablepattern is not evaluated
.*               SystemVerilog Patterns wildcardpattern is not evaluated
tagged Valid .x  SystemVerilog Patterns taggedpattern is not evaluated
tagged None      SystemVerilog Patterns taggedpattern is not evaluated
'{.a, .b}        SystemVerilog Patterns structurepattern is not evaluated
v matches tagged Valid .x ? x + 1 : 0 SystemVerilog matchesexpression is not evaluated
inferred: any


<a id="T08.14.6"></a>
## 08.14.6 · Its editions are data

Pattern matching arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions, 1800-2005 to 1800-2023,
and none of Verilog's, each with every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
singleton = store.Instances.Expressions.SystemVerilog.Patterns
kinds = {branch.schema: branch.name for branch in store.registered(Patterns.union).branches}
for designation, edition in Standards.standards(singleton).items():
    print(f"{designation:19}", f"{Standards.tags(edition)[1]:19}", len(Standards.kinds(edition)), "kinds")
print("each:", ", ".join(kinds[k] for k in Standards.kinds(Standards.find(singleton, "SystemVerilog-2005"))))
print("Verilog-2005:", "covered" if Standards.find(singleton, "Verilog-2005") else "not covered")

IEEE Std 1800-2005  SystemVerilog-2005  6 kinds
IEEE Std 1800-2009  SystemVerilog-2009  6 kinds
IEEE Std 1800-2012  SystemVerilog-2012  6 kinds
IEEE Std 1800-2017  SystemVerilog-2017  6 kinds
IEEE Std 1800-2023  SystemVerilog-2023  6 kinds
each: variablepattern, wildcardpattern, taggedpattern, structurepattern, patternmember, parenthesizedpattern
Verilog-2005: not covered


## What you learned

- **A pattern is a term of `SystemVerilog Patterns`**: a variable, a wildcard, a tagged, structure or parenthesized
  pattern, nesting; a constant pattern is an expression, which its `Term` lists.
- **A pattern binds its variables, through what holds it**: a `matches` binds them in its conditional's consequence and
  in the conditions after it, and nowhere else.
- **It is validated as any term**: a structure pattern in order or by name, not both, and its constants' names bound.
- **It is not evaluated**: each kind, and the `matches` that holds it, refused by name; its editions are IEEE 1800's.

Next, sequences and properties: [SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb).

<!-- nav -->
[← 8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb) · [Home](../../README.md) · [8.15 · SystemVerilog Assertions →](08_15_SystemVerilog_Assertions.ipynb)